We are going to compute the eigenvalues and eigenvectors of the dataset.

In [ ]:
from pathlib import Path
from graph2mat4abn.tools.tbplas_tools import add_orbitals, add_hopping_terms, extract_onsites_from_coo, extract_hoppings_from_coo, compute_k_len, select_kpath
from scipy.sparse import save_npz
from tqdm import tqdm

import sisl
import tbplas as tb
import numpy as np

In [ ]:
# Create directory

eigen_dir = Path("../dataset_eigen") # Assuming this notebook is in ./graph2mat4abn/notebooks
eigen_dir.mkdir(exist_ok=True)

# Load the paths to all structures
dataset_dir = Path("../dataset")
use_only_x_atoms = ["2", "8"]
subsets = [subset for subset in dataset_dir.glob("*/") if subset.parts[-1].split("_")[-2] in use_only_x_atoms]
subsets

In [ ]:
# Iterate for each subset of SHARE_OUTPUTS_X_ATOMS, then through each structure to finally save each diagonalization.
for subset in tqdm(subsets):
    for structure_path in tqdm(subset.glob("*/")):

        file = sisl.get_sile(structure_path / "aiida.fdf")
        geometry = file.read_geometry()
        positions = geometry.xyz # Angstroms
        labels = [[orb.name() for orb in atom] for atom in geometry.atoms]
        hamiltonian_coo = file.read_hamiltonian().tocsr().tocoo()

        n_atoms = int(structure_path.parts[-2].split("_")[-2])
        vectors = geometry.cell
        cell = tb.PrimitiveCell(vectors, unit=tb.ANG)


        # Construct tbplas cell
        onsites = extract_onsites_from_coo(hamiltonian_coo)
        add_orbitals(cell, positions, onsites, labels)

        iscs, orbs_in, orbs_out, hoppings = extract_hoppings_from_coo(hamiltonian_coo, n_atoms, geometry)
        add_hopping_terms(cell, iscs, orbs_in, orbs_out, hoppings)

        # Construct tbplas overlap
        overlap_coo = file.read_overlap().tocsr().tocoo()
        overlap_cell = tb.PrimitiveCell(cell.lat_vec, cell.origin, 1.0)

        onsites = extract_onsites_from_coo(overlap_coo)
        add_orbitals(overlap_cell, positions, onsites, labels)

        iscs, orbs_in, orbs_out, hoppings = extract_hoppings_from_coo(overlap_coo, n_atoms, geometry)
        add_hopping_terms(overlap_cell, iscs, orbs_in, orbs_out, hoppings)


        # Compute the paths
        k_path, k_idx, k_label = select_kpath(n_atoms, cell, n_kpoints=20, structure=None)
        

        # Compute the bands
        solver = tb.DiagSolver(cell, overlap_cell)
        solver.config.k_points = k_path
        bands, states = solver.calc_states()
        k_len = compute_k_len(k_path)
        solver.config.prefix = "bands"
        
        # Save the results
        save_dir = eigen_dir / structure_path.parts[-2] / structure_path.parts[-1]
        save_dir.mkdir(parents=True, exist_ok=True)
        np.savez(save_dir/"bands.npz", path=str(structure_path), bands=bands)
        np.savez(save_dir/"k_path.npz", path=str(structure_path), k_path=k_path, k_idx=k_idx, k_label=k_label, k_len=k_len)
        np.savez(save_dir/"states.npz", path=str(structure_path), states=states)
        save_npz(save_dir/"overlap.npz", overlap_coo)
        


# Test load and matmul

In [1]:
from scipy.sparse import load_npz
from pathlib import Path
import sisl
import numpy as np

path = Path("../dataset_eigen/SHARE_OUTPUTS_2_ATOMS/6d1a-578f-478c-a320-d8be012039e6")
geometry = sisl.get_sile(Path(str(path).replace("dataset_eigen", "dataset")) / "aiida.fdf").read_geometry()
cell = geometry.cell

bands = np.load(path/"bands.npz")["bands"]
states = np.load(path/"states.npz")["states"]
overlap_coo = load_npz(path/"overlap.npz")
k = np.load(path/"k_path.npz")
k_path = k["k_path"]
k_idx = k["k_idx"]
k_label = k["k_label"]
k_len = k["k_len"]



In [2]:
print(bands.shape)
print(states.shape)
print(overlap_coo.shape)
print(k_path.shape)

(61, 26)
(61, 26, 26)
(26, 3250)
(61, 3)


Given the overlap matrix, we must compute the "TIM" of the overlap matrix at the specific k-point.

In [ ]:
from graph2mat4abn.tools.tools import reduced_coord, reconstruct_tim_from_coo
import torch

overlap_tim = reconstruct_tim_from_coo(k_path, overlap_coo, geometry, cell)
print(overlap_tim.shape)
print(states.shape) # [n_kpoints, n_bands, n_orbs]
print(bands.shape) # [n_kpoints, n_bands]
print(k_path.shape)
print((overlap_tim @ states).shape)
residual = torch.tensor(bands[:, :, None] * (overlap_tim @ states))
print(residual.shape)

(61, 26, 26)
(61, 26, 26)
(61, 26)
(61, 3)
(61, 26, 26)
torch.Size([61, 26, 26])


In [ ]:
torch.sum(residual.conj() * residual).real #This gives the sum of the norm |r|^2 for all bands and kpoints (exactly what we need lol)
states_bands = [(residual[0,i].conj() @ residual[0,i]).real for i in range(residual.shape[1])]
sum(states_bands)
torch.sum(residual[0].conj() * residual[0]).real


tensor(2.0350e+09, dtype=torch.float64)

# Test reconstruct_tim_from_coo when batching

In [ ]:
overlap_tim = reconstruct_tim_from_coo(k_path, overlap_coo, geometry, cell)
overlap_coo

TypeError: 'coo_matrix' object is not subscriptable